# Phenotype residualization

Input: `03_round2_1000g_filter.ipynb`'s ancestry-filtered keep-list and `04_final_pca.ipynb`'s covariate PCs (person IDs, `01_ancestry_filtering` -- two-round 1000G-referenced classification) and a phenotype list TSV. Output: one `FID IID Y` file per phenotype x transform x covariate-set combination, matching `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected format.

Protocol:
1. drop physiologically-implausible values (`plausible_min`/`plausible_max` in the phenotype list TSV, `filter_plausible_range()`) -- catches data-entry/unit errors, not just statistical outliers
2. residualize `phenotype ~ covariates` (order matters from here, see Kemper et al. 2021 Methods)
3. trim residuals > 5 SD from the mean
4. standardize surviving residuals to mean 0, var 1, within each sex

Covariates: sex-at-birth + age always included. `build_covariate_sets()` is a nested staircase, not independently-toggled combinations — `base` (age only) -> `base_pcs` (+ PCs) -> `base_pcs_zip3` (+ 3-digit zip factor) -> `base_pcs_zip3_ses` (+ SES vars from `zip3_ses_map`). Every one of those also gets crossed with raw vs. rank-inverse-normal-transformed.

## Setup

**Run this cell first in a freshly restarted kernel.** If a package (e.g. `rlang`) already got auto-loaded from the system library before this cell runs -- which can happen just from kernel startup -- `.libPaths()` can't retroactively swap it for the newer version installed here; R won't unload/replace an already-attached namespace mid-session. Symptom: `namespace 'rlang' 1.1.6 is already loaded, but >= 1.1.7 is required`. Fix is a full kernel restart, not re-running cells in the same session.

In [ ]:
required_pkgs <- c("dplyr", "tidyr", "readr", "stringr", "e1071", "bigrquery", "allofus")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(tidyr)
library(readr)
library(stringr)
library(bigrquery)
library(allofus)
source("../../scripts/local/residualize_lib.R")   # transform, residualize, export -- shared with the fake-data test

con <- aou_connect()   # BigQuery connection to the CDR, used by pull_phenotype() below

## Inputs

- `KEEP_LIST_PATH`: `03_round2_1000g_filter.ipynb`'s output (two-round 1000G-referenced classification, `01_ancestry_filtering`) — person IDs passing ancestry filtering.
- `PHENO_LIST_PATH`: TSV describing which phenotypes to pull (see schema below).
- `PC_PATH`: `04_final_pca.ipynb`'s output — `IID`, `PC1`..`PC10` fit *within* this exact `SAMPLE_SET`'s own final members (not reused from a broader fit).
- `ZIP3_SES_TABLE`: AoU's `zip3_ses_map` — exact join path (likely via `observation`, not directly on `person`) still needs confirming against the real workbench, flagged below.

Phenotype list TSV schema (`PHENO_LIST_PATH`):

| column | purpose |
|---|---|
| `phenotype_name` | label used in output filenames/diagnostics |
| `source` | `measurement` / `measurement_unit_normalized` / `survey` / `survey_composite` / `derived_ratio` / `condition` / `fitbit` -- picks the extraction path (`measurement`/`survey`/`measurement_unit_normalized` are all generic in `pull_phenotype()`; `derived_ratio` (`waist_hip_ratio`) and `survey_composite` (`alcohol_audit_c_score`) are each implemented as their own cell instead, since neither fits `pull_phenotype()`'s one-concept_id-in/one-value-out shape; `condition`/`fitbit` aren't wired up yet) |
| `concept_id` | AoU concept ID (comma-separated for multiple) |
| `value_field` | optional -- which field holds the numeric value |
| `plausible_min` / `plausible_max` | physiologically-plausible range (original units) -- values outside are dropped before any modeling, see `filter_plausible_range()` in `residualize_lib.R` |
| `notes` | free text, not used programmatically |

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")

# Must match whatever CDR_VERSION 01_ancestry_filtering's notebooks were
# actually run with -- this pipeline stage reads their output.
CDR_VERSION <- "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR <- "covariance_v9"

ANCESTRY_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, PROJECT_DIR, "01_ancestry_filtering")
PHENOTYPE_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, PROJECT_DIR, "02_phenotype")

# Which of 04_final_pca.ipynb's 5 sample sets to residualize phenotypes for.
# Set SAMPLE_SET and rerun this notebook once per sample set (same manual-rerun
# convention as CDR_VERSION) -- OUT_DIR/MODELING_TABLE_DIR/RAW_PHENO_CACHE_DIR
# below are all tagged by SAMPLE_SET so all 5 runs' outputs coexist rather than
# overwrite each other. prob_tag must match 03_round2_1000g_filter.ipynb's own
# SAMPLE_SETS dict (ellipsoid_threshold -> prob_tag derivation,
# f"p{threshold*100:g}", or "unfiltered" for eur_unfiltered) for that sample
# set -- keep these two in sync by hand, since this notebook doesn't import that
# one's SAMPLE_SETS dict directly.
SAMPLE_SET <- "eur"   # <-- change this and rerun for each of the 5 sample sets
sample_set_cfg <- switch(SAMPLE_SET,
  eur =            list(prob_tag = "p99.9999"),
  eur_stringent =  list(prob_tag = "p99"),
  eur_loose =      list(prob_tag = "p99.999999"),
  eur_unfiltered = list(prob_tag = "unfiltered"),
  afr =            list(prob_tag = "p99.9")
)

# single ancestry panel now, not per-BASE_GROUP -- 03_round2_1000g_filter.ipynb's
# keep-list and 04_final_pca.ipynb's per-SAMPLE_SET covariate PCs both live
# under ancestry_panel/final_pca/
FINAL_PCA_DIR <- file.path(ANCESTRY_BUCKET_DIR, "ancestry_panel", "final_pca")

KEEP_LIST_PATH <- file.path(FINAL_PCA_DIR, paste0("final_keep_ids_", SAMPLE_SET, "_", sample_set_cfg$prob_tag, ".txt"))   # 03_round2_1000g_filter.ipynb output, one person_id per line
PHENO_LIST_PATH <- "../../docs/phenotype_list.tsv"  # starter anthropometric/metabolic panel -- see its header for concept_id confirmation status
PC_PATH <- file.path(FINAL_PCA_DIR, SAMPLE_SET, paste0("final_pca_pc_covariates_", SAMPLE_SET, ".txt"))   # 04_final_pca.ipynb output -- PC1-PC10 fit within this exact sample set
OUT_DIR <- file.path(PHENOTYPE_BUCKET_DIR, SAMPLE_SET, "residualized")          # where the FID/IID/Y files get written
MODELING_TABLE_DIR <- file.path(PHENOTYPE_BUCKET_DIR, SAMPLE_SET, "modeling_tables")   # one neat TSV per phenotype (prepare_modeling_tables()'s
                                       # output) -- lives in the workspace bucket (per root README's bucket
                                       # layout), not local disk: this is what lets the residualization
                                       # procedure itself get retuned later without re-pulling from BigQuery
REFERENCE_DATE <- "2024-01-01"   # arbitrary fixed date -- adjust to match the CDR version's actual data cutoff, same convention as 01_query_filter_check.ipynb
RAW_PHENO_CACHE_DIR <- file.path(PHENOTYPE_BUCKET_DIR, SAMPLE_SET, "raw_pheno_cache")  # raw pull_phenotype() output per phenotype, cached so repeated
                                       # runs while iterating on residualize_lib.R don't re-hit BigQuery -- per
                                       # sample set since keep_ids differs, so a cached pull for one sample set
                                       # can't be reused for another (the keep_ids filter is applied before caching)

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(MODELING_TABLE_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(RAW_PHENO_CACHE_DIR, recursive = TRUE, showWarnings = FALSE)

## Read the ID list and phenotype list

`keep_ids`: everyone downstream gets restricted to this set before anything else -- residualization, transforms, and diagnostics are all computed within this cohort only, not the full AoU population.

In [ ]:
keep_ids_raw <- read_lines(KEEP_LIST_PATH)
# 03_round2_1000g_filter.ipynb's keep-list is plink-style: either one ID per line, or "FID IID"
# space-separated -- take the last whitespace-separated field either way
keep_ids <- str_trim(keep_ids_raw) %>% str_split(" +") %>% sapply(function(x) tail(x, 1))

pheno_list_all <- read_tsv(PHENO_LIST_PATH, col_types = cols(.default = "c"))
stopifnot(all(c("phenotype_name", "source", "concept_id", "plausible_min", "plausible_max") %in% names(pheno_list_all)))

# UNCONFIRMED concept_ids (currently the 3 lipid rows -- see phenotype_list.tsv's
# notes column) aren't real AoU concept IDs and can't be queried; excluded here
# rather than left to fail deep inside pull_phenotype() with an opaque BigQuery
# error. Confirm the concept_id against the AoU Data Browser and edit
# phenotype_list.tsv to include those phenotypes in a run.
unconfirmed <- pheno_list_all$phenotype_name[pheno_list_all$concept_id == "UNCONFIRMED"]
if (length(unconfirmed) > 0) {
  message("Skipping phenotypes with UNCONFIRMED concept_id: ", paste(unconfirmed, collapse = ", "))
}
pheno_list <- pheno_list_all %>% filter(concept_id != "UNCONFIRMED")
pheno_list

## Pull phenotype + covariate table

`pull_phenotype()` handles `source == "measurement"` via `allofus::aou_sql()` (most recent value per person, age computed from a single fixed `REFERENCE_DATE` rather than per-measurement). Sex is `person.sex_at_birth_concept_id` (a direct AoU-specific column -- distinct from `person.gender_concept_id`, which is gender identity). `source == "survey"` (`cigarettes_per_day`) uses the identical most-recent-answer-per-person shape, just against `{CDR}.observation`/`observation_concept_id`/`observation_date` instead -- `cigarettes_per_day`'s `concept_id` holds 2 comma-separated PPI concept_ids (AoU branches the question by current-vs-former smoking status, so a person answers at most one), and the shared `IN (...)` + `ROW_NUMBER()` logic naturally coalesces across both. `source == "measurement_unit_normalized"` (`hemoglobin`) is the same most-recent-per-person `{CDR}.measurement` pull as `measurement`, plus `unit_concept_id` carried along and used to divide by 10 wherever it's 8636 ("gram per liter") -- `unit_source_value`'s free-text label turned out unreliable even for its own dominant group (see `phenotype_list.tsv`'s notes and `01_query_filter_check.ipynb`'s hemoglobin appendix for the full investigation). `pull_covariates()`'s zip3/SES join uses `zip3_ses_map`, keyed off a masked zip-code `observation` row (AoU privacy-protects exact zip, only the 3-digit prefix survives, marked with a `*` in `value_as_string`). `survey_composite`/`condition`/`fitbit`/`derived_ratio` phenotype sources still aren't wired up here -- `pull_phenotype()` throws a clear error on any of them rather than silently doing nothing, and `prepare_modeling_tables()` catches that per-phenotype and skips it (see below) rather than crashing the whole run. `waist_hip_ratio` (`derived_ratio`) and `alcohol_audit_c_score` (`survey_composite`) are the two exceptions with an actual implementation, each built as its own cell further down instead of inside `pull_phenotype()`, since neither fits that function's one-concept_id-in, one-value-out shape.

Everything below this cell (transform, covariate-set configs, residualization, export, diagnostics) doesn't depend on these specifics and is fully generic once `pull_phenotype()` returns a table shaped `person_id, phenotype, age, sex_at_birth`.

In [ ]:
pull_phenotype <- function(row, keep_ids) {
  cache_path <- file.path(RAW_PHENO_CACHE_DIR, paste0(row$phenotype_name, ".tsv"))
  if (file.exists(cache_path)) {
    return(read_tsv(cache_path, col_types = cols(person_id = "c", phenotype = "d", age = "d", sex_at_birth = "c")))
  }

  if (!(row$source %in% c("measurement", "survey", "measurement_unit_normalized"))) {
    stop(sprintf(
      "pull_phenotype(): source '%s' not implemented -- only 'measurement'/'survey'/'measurement_unit_normalized' are wired up so far",
      row$source
    ))
  }

  if (!grepl("^[0-9]+(,[0-9]+)*$", row$concept_id)) {
    stop(sprintf(
      "pull_phenotype(): '%s' has concept_id '%s', not a valid comma-separated numeric AoU concept ID -- confirm it against the AoU Data Browser and fix docs/phenotype_list.tsv before running this phenotype",
      row$phenotype_name, row$concept_id
    ))
  }

  if (row$source == "measurement_unit_normalized") {
    # hemoglobin (LOINC 718-7, concept_id 3000963): unit_source_value's free-text label
    # is unreliable, even for its own dominant group -- confirmed via
    # 01_query_filter_check.ipynb's hemoglobin appendix: the group literally labeled
    # "g/dL" (the large majority of this concept's data) has its own median value of
    # 124, i.e. actually g/L. unit_concept_id is the reliable signal instead -- every
    # sizeable group shares unit_concept_id 8636 ("gram per liter") regardless of what
    # the source-text label says, so divide by 10 wherever unit_concept_id = 8636 and
    # leave unit_concept_id 0/NA (genuinely unknown unit) as reported. Confirmed
    # against the real CDR: drops this concept's implausible-range exclusion rate from
    # ~90% to ~1% on the full unrestricted cohort.
    GRAM_PER_LITER_UNIT_CONCEPT_ID <- 8636

    query <- sprintf("
      WITH demographics AS (
        SELECT
          person_id,
          CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
          CASE
            WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
            WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
            ELSE 'Other'
          END AS sex_at_birth
        FROM {CDR}.person
      ),
      measurements AS (
        SELECT
          person_id,
          value_as_number,
          unit_concept_id,
          ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
        FROM {CDR}.measurement
        WHERE measurement_concept_id IN (%s)
          AND value_as_number IS NOT NULL
      )
      SELECT d.person_id, m.value_as_number, m.unit_concept_id, d.age, d.sex_at_birth
      FROM demographics d
      INNER JOIN measurements m ON d.person_id = m.person_id
      WHERE m.rn = 1
    ", REFERENCE_DATE, row$concept_id)

    pheno_df <- aou_sql(query) %>%
      collect() %>%
      mutate(
        person_id = as.character(person_id),
        phenotype = ifelse(unit_concept_id == GRAM_PER_LITER_UNIT_CONCEPT_ID, value_as_number / 10, value_as_number)
      ) %>%
      select(person_id, phenotype, age, sex_at_birth) %>%
      filter(person_id %in% keep_ids)

    write_tsv(pheno_df, cache_path)
    return(pheno_df)
  }

  # source == "measurement": most-recent-value-per-person from {CDR}.measurement.
  # source == "survey": identical shape, but from {CDR}.observation's value_as_number --
  # cigarettes_per_day's concept_id holds two comma-separated PPI concept_ids (current-
  # smoker daily count, lifetime-average daily count); AoU's branching survey logic means
  # a person answers at most one, so this IN (...) + ROW_NUMBER()-most-recent approach
  # naturally coalesces across both. "Don't know"/"Skip"/"prefer not to answer" responses
  # are value_as_concept_id-coded, not numeric, so the value_as_number IS NOT NULL filter
  # already excludes them without any extra lookup.
  source_table <- if (row$source == "measurement") "measurement" else "observation"
  concept_col <- if (row$source == "measurement") "measurement_concept_id" else "observation_concept_id"
  date_col <- if (row$source == "measurement") "measurement_date" else "observation_date"

  # DATE_DIFF returns INT64; bigrquery collects bare INT64 columns as bit64::integer64,
  # which lm() silently mis-coerces into a degenerate fit rather than erroring -- cast to
  # FLOAT64 in the query so age collects as a plain double
  query <- sprintf("
    WITH demographics AS (
      SELECT
        person_id,
        CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
        CASE
          WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
          WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
          ELSE 'Other'
        END AS sex_at_birth
      FROM {CDR}.person
    ),
    answers AS (
      SELECT
        person_id,
        value_as_number AS phenotype,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY %s DESC) AS rn
      FROM {CDR}.%s
      WHERE %s IN (%s)
        AND value_as_number IS NOT NULL
    )
    SELECT d.person_id, a.phenotype, d.age, d.sex_at_birth
    FROM demographics d
    INNER JOIN answers a ON d.person_id = a.person_id
    WHERE a.rn = 1
  ", REFERENCE_DATE, date_col, source_table, concept_col, row$concept_id)

  pheno_df <- aou_sql(query) %>%
    collect() %>%   # pull out of BigQuery before any local joins/filters -- a
                     # lazy remote tbl can't be left_join()'d against a local one
    mutate(person_id = as.character(person_id)) %>%
    filter(person_id %in% keep_ids)

  write_tsv(pheno_df, cache_path)   # cache the raw pull -- avoids re-hitting BigQuery on every
                                     # downstream iteration; delete the file to force a refresh
  pheno_df
}

pull_covariates <- function(keep_ids) {
  pcs <- read_table(PC_PATH, col_types = cols(.default = "d", IID = "c")) %>%
    rename(person_id = IID) %>%
    filter(person_id %in% keep_ids)

  # zip3 comes from a masked (privacy-protected) zip-code observation --
  # AoU marks these rows with a "*" in value_as_string; the first 3 characters
  # are the 3-digit zip prefix, joined against the CDR-provided zip3_ses_map.
  # median_income/poverty/deprivation_index cast to FLOAT64 for the same reason
  # as age in pull_phenotype() -- an unqualified INT64 column collects as
  # bit64::integer64, which lm() silently mis-coerces rather than erroring
  zip3_ses_query <- "
    SELECT
      o.person_id,
      z.zip3,
      CAST(z.median_income AS FLOAT64) AS median_income,
      CAST(z.fraction_poverty AS FLOAT64) AS poverty,
      CAST(z.deprivation_index AS FLOAT64) AS deprivation_index
    FROM (
      SELECT person_id, CAST(SUBSTR(value_as_string, 1, 3) AS INT64) AS zip3
      FROM {CDR}.observation
      WHERE STRPOS(value_as_string, '*') > 0
    ) o
    INNER JOIN {CDR}.zip3_ses_map z ON o.zip3 = z.zip3
  "
  zip3_ses <- aou_sql(zip3_ses_query) %>%
    collect() %>%
    mutate(person_id = as.character(person_id)) %>%
    filter(person_id %in% keep_ids)

  zip3 <- zip3_ses %>% transmute(person_id, zip3 = as.character(zip3))
  ses <- zip3_ses %>% select(person_id, median_income, poverty, deprivation_index)

  list(pcs = pcs, zip3 = zip3, ses = ses)
}

## Optional skew-reducing transform

Rank-based inverse-normal transform (`inverse_normal_transform()` in `residualize_lib.R`) -- chosen over log/Box-Cox since it doesn't assume a particular skew direction or require positive values, so it works uniformly across an arbitrary phenotype list without per-phenotype tuning. Both `raw` and `invnorm` variants get carried through the rest of the pipeline; skewness before/after is reported so it's clear whether the transform actually helped for each phenotype, rather than applying it blindly. See `02_phenotype/notebooks/local/test_residualize_fake_data.ipynb` for a worked example on synthetic data showing this diagnostic actually catching a real skew reduction.

## Covariate-set configs

`build_covariate_sets()` (in `residualize_lib.R`) returns a named list of formula RHS vectors. `sex_at_birth` is handled separately (it's the stratification variable for step 3, not a residualization covariate -- see `residualize_phenotype()` in the lib), so it's not in these formulas.

In [ ]:
pc_cols <- paste0("PC", 1:5)   # top 5 of the 10 04_final_pca.ipynb writes -- beyond that isn't considered informative for this cohort
covariate_sets <- build_covariate_sets(pc_cols)
names(covariate_sets)

## Prepare modeling tables

`prepare_modeling_tables()` (in `residualize_lib.R`) is the only cell below
that hits BigQuery -- pulls each phenotype, applies the plausible-range
filter, joins PCs/zip3/SES, adds the invnorm variant, and writes one neat
TSV per phenotype to `MODELING_TABLE_DIR` (`<phenotype_name>.tsv`:
`person_id, phenotype, phenotype__invnorm, age, sex_at_birth`, plus every
covariate column). Re-run this cell only when the phenotype list,
keep-list, or covariate pulls themselves change.

If `pull_phenotype()` throws for a given row (wrong/unimplemented `source`, a `concept_id` that doesn't actually exist in this CDR version, a transient BigQuery error), that phenotype is skipped -- `message()`-printed, recorded in `range_summary_table` below with `status == "skipped: pull_phenotype() failed"` -- rather than taking down every other phenotype's pull too. `pull_covariates()` is deliberately *not* given the same treatment: it runs once per phenotype but is the same call every time, so a failure there is systemic, not phenotype-specific, and should stop the run loudly.

In [ ]:
prep <- prepare_modeling_tables(
  pheno_list, keep_ids, pull_phenotype, pull_covariates, MODELING_TABLE_DIR
)
prep$range_summary_table  # per phenotype: N excluded by the plausible-range filter, before any modeling

## Derived phenotype: waist/hip ratio

`waist_hip_ratio` (`source == "derived_ratio"` in `phenotype_list.tsv`)
isn't a single pulled measurement -- `pull_phenotype()` throws on it
immediately (unimplemented source), which `prepare_modeling_tables()`'s
per-phenotype error handling turns into a skip, not a crash. This cell
builds its modeling table directly instead, by combining the
`waist_circumference`/`hip_circumference` tables the "Prepare modeling
tables" cell above already wrote (no new BigQuery calls) -- so it must run
after that cell, before "Run residualization" below.

**Caveat:** `waist`/`hip` are each independently "most recent value as of
`REFERENCE_DATE`" (same as every other phenotype here), joined on
`person_id` -- not matched to the same visit/date. In practice AoU's
Physical Measurements module tends to collect both together, so this is
usually fine, but if the ratio's distribution looks off, mismatched-visit
pairing is the first thing to check.

In [ ]:
waist_path <- file.path(MODELING_TABLE_DIR, "waist_circumference.tsv")
hip_path <- file.path(MODELING_TABLE_DIR, "hip_circumference.tsv")

if (file.exists(waist_path) && file.exists(hip_path)) {
  waist <- read_tsv(waist_path, show_col_types = FALSE) %>% select(person_id, waist = phenotype)
  hip <- read_tsv(hip_path, show_col_types = FALSE) %>% select(person_id, hip = phenotype, age, sex_at_birth)

  ratio_row <- pheno_list_all %>% filter(phenotype_name == "waist_hip_ratio")
  stopifnot(nrow(ratio_row) == 1)

  whr_df <- waist %>%
    inner_join(hip, by = "person_id") %>%
    mutate(phenotype = waist / hip) %>%
    select(person_id, phenotype, age, sex_at_birth)

  range_result <- filter_plausible_range(
    whr_df, "phenotype", as.numeric(ratio_row$plausible_min), as.numeric(ratio_row$plausible_max)
  )
  whr_df <- range_result$data

  covars <- pull_covariates(keep_ids)
  whr_df <- whr_df %>%
    left_join(covars$pcs, by = "person_id") %>%
    left_join(covars$zip3, by = "person_id") %>%
    left_join(covars$ses, by = "person_id") %>%
    add_transformed_variant("phenotype")

  write_tsv(whr_df, file.path(MODELING_TABLE_DIR, "waist_hip_ratio.tsv"))
  message(sprintf(
    "waist_hip_ratio: %d excluded as implausible, %d written to %s",
    range_result$n_excluded, nrow(whr_df), file.path(MODELING_TABLE_DIR, "waist_hip_ratio.tsv")
  ))
} else {
  message(
    "Skipping waist_hip_ratio: waist_circumference.tsv and/or hip_circumference.tsv not found in ",
    "MODELING_TABLE_DIR -- run 'Prepare modeling tables' first, and confirm both concept_ids in ",
    "phenotype_list.tsv if either was skipped as UNCONFIRMED or failed to pull"
  )
}

## Derived phenotype: alcohol AUDIT-C score

`alcohol_audit_c_score` (`source == "survey_composite"` in `phenotype_list.tsv`)
is a sum of 3 separate survey answers, not a single pulled value --
`pull_phenotype()` throws on it immediately (unimplemented source), same as
`waist_hip_ratio` above. Unlike `waist_hip_ratio`, this doesn't depend on
any other phenotype's modeling table, so it can run independently of
"Prepare modeling tables" above (it still needs `keep_ids`/`pull_covariates()`
from earlier cells).

**Concept_ids and scoring, confirmed via `01_query_filter_check.ipynb`'s
lifestyle appendix (see that notebook for the full investigation):** the
PPI vocabulary's own concept_ids for the frequency/quantity items
(`Alcohol_DrinkFrequencyPastYear`/`Alcohol_AverageDailyDrinkCount`) turned
out non-standard and entirely unpopulated in `observation_concept_id` --
same "Maps to" mismatch pattern as `hip_circumference`'s LOINC bug. The
concept_ids below are what those actually map to (`FREQ_CONCEPT_ID`/
`QUANTITY_CONCEPT_ID`), plus the binge item (`BINGE_CONCEPT_ID`), which was
already standard. All 3 answers are `value_as_concept_id`-coded (a chosen
answer-option concept), not plain numbers -- each item's answer options
were verified directly against real answer-option counts to map cleanly
onto the standard 0-4 AUDIT-C scale; `*_score_map` below encodes that
lookup. Any answer not in a given map (`PMI: Skip`, `Don't know`, and a
handful of stray legacy-vocabulary codes on the frequency item -- <0.01%
of responses) scores `NA`. The composite (`phenotype`) is only defined
when all 3 items are answered -- `NA` propagates through the sum by
design, since a partial AUDIT-C score isn't a meaningful value.

In [ ]:
FREQ_CONCEPT_ID <- 40771103      # "How often do you have a drink containing alcohol" -- AUDIT-C item 1
QUANTITY_CONCEPT_ID <- 40771104  # "...how many standard drinks... on a typical day" -- AUDIT-C item 2
BINGE_CONCEPT_ID <- 1586213      # "Alcohol: 6 or More Drinks Occurrence" -- AUDIT-C item 3

# answer-concept -> standard 0-4 AUDIT-C item score; anything not listed (PMI: Skip,
# Don't know, stray legacy codes) is absent from the map and scores NA below
freq_score_map <- c(`45876662` = 0, `45879058` = 1, `45885058` = 2, `45877711` = 3, `45885059` = 4)
quantity_score_map <- c(`45882591` = 0, `45883428` = 1, `45877926` = 2, `45877712` = 3, `45879059` = 4)
binge_score_map <- c(`1585634` = 0, `45885060` = 1, `45879060` = 2, `45877713` = 3, `45879676` = 4)

pull_audit_c_item <- function(concept_id, score_map, col_name) {
  query <- sprintf("
    SELECT
      person_id,
      value_as_concept_id,
      ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY observation_date DESC) AS rn
    FROM {CDR}.observation
    WHERE observation_concept_id = %d
  ", concept_id)

  aou_sql(query) %>%
    collect() %>%
    filter(rn == 1) %>%
    transmute(
      person_id = as.character(person_id),
      !!col_name := unname(score_map[as.character(value_as_concept_id)])
    )
}

audit_c_demographics <- aou_sql(sprintf("
  SELECT
    person_id,
    CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
    CASE
      WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
      WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
      ELSE 'Other'
    END AS sex_at_birth
  FROM {CDR}.person
", REFERENCE_DATE)) %>% collect() %>% mutate(person_id = as.character(person_id))

audit_c_row <- pheno_list_all %>% filter(phenotype_name == "alcohol_audit_c_score")
stopifnot(nrow(audit_c_row) == 1)

audit_c_df <- pull_audit_c_item(FREQ_CONCEPT_ID, freq_score_map, "freq_score") %>%
  full_join(pull_audit_c_item(QUANTITY_CONCEPT_ID, quantity_score_map, "quantity_score"), by = "person_id") %>%
  full_join(pull_audit_c_item(BINGE_CONCEPT_ID, binge_score_map, "binge_score"), by = "person_id") %>%
  mutate(phenotype = freq_score + quantity_score + binge_score) %>%   # NA propagates if any item is missing/unscored, by design
  inner_join(audit_c_demographics, by = "person_id") %>%
  filter(person_id %in% keep_ids)

range_result <- filter_plausible_range(
  audit_c_df, "phenotype", as.numeric(audit_c_row$plausible_min), as.numeric(audit_c_row$plausible_max)
)
audit_c_df <- range_result$data

covars <- pull_covariates(keep_ids)
audit_c_df <- audit_c_df %>%
  select(person_id, phenotype, age, sex_at_birth) %>%
  left_join(covars$pcs, by = "person_id") %>%
  left_join(covars$zip3, by = "person_id") %>%
  left_join(covars$ses, by = "person_id") %>%
  add_transformed_variant("phenotype")

write_tsv(audit_c_df, file.path(MODELING_TABLE_DIR, "alcohol_audit_c_score.tsv"))
message(sprintf(
  "alcohol_audit_c_score: %d excluded as implausible, %d written to %s",
  range_result$n_excluded, nrow(audit_c_df), file.path(MODELING_TABLE_DIR, "alcohol_audit_c_score.tsv")
))

## Derived phenotypes: SDOH composite scores (PSS, social support, loneliness, discrimination)

Five `source == "survey_composite"` phenotypes from AoU's `sdoh_*`-prefixed
survey module (`sdoh_environment_concept_search.ipynb` has the full concept
discovery) -- each a sum of several `value_as_concept_id`-coded items, same
shape as `alcohol_audit_c_score` above, just more items per scale (8-10
instead of 3). None have a pre-computed composite under their own "instrument"
concept_id in this CDR (checked directly -- zero rows), so all four are summed
here from real item-level answers.

`compute_sdoh_composite()` below is the shared shape: pull every item's
most-recent answer per person, map `value_as_concept_id` -> integer score via
a scale-specific lookup, optionally reverse-score specific items, then sum --
**requiring every item in the scale to be answered** (no partial-scale
imputation, same convention as `alcohol_audit_c_score`'s "NA propagates if any
item is missing" choice).

- **PSS** (Cohen's Perceived Stress Scale-10, `cpss_1`..`cpss_10`): Likert map
  and the 4 reverse-scored items (`cpss_4`/`5`/`7`/`8`) confirmed against real
  `{CDR}.concept` labels.
- **Social support** (RAND MOS-SS8, `mos_ss_1`..`mos_ss_8`): Likert map
  confirmed the same way.
- **Loneliness** (UCLA-LS8, `ucla_ls8_*`, 8 non-sequential item numbers):
  confirmed 4-point in this CDR (Never/Rarely/Sometimes/Often), not the
  published instrument's 3-point scale -- flagged since it's a deviation from
  UCLA-LS norms if ever compared to published cutoffs.
- **Everyday Discrimination** (`eds_1`..`eds_9`): **not yet confirmed** the
  same way -- the Likert map below is text-matched against the published
  6-point EDS response scale (Never .. Almost every day) rather than
  hardcoded concept_ids from a real query, and will warn loudly on any
  unmatched label instead of silently mis-scoring. Run the diagnostic cell
  first and inspect its output before trusting `eds_score`.

In [ ]:
# Shared shape for every SDOH composite below: most-recent answer per item per
# person, map value_as_concept_id -> integer score, optionally reverse-score
# specific items, sum -- requiring every item answered (partial scales -> NA,
# not imputed).
compute_sdoh_composite <- function(item_concept_ids, score_map, reverse_items = character(0), reverse_max = 4) {
  items_query <- sprintf("
    SELECT person_id, observation_source_concept_id, value_as_concept_id,
           ROW_NUMBER() OVER (
             PARTITION BY person_id, observation_source_concept_id
             ORDER BY observation_date DESC
           ) AS rn
    FROM {CDR}.observation
    WHERE observation_source_concept_id IN (%s)
  ", paste(item_concept_ids, collapse = ","))

  items_df <- aou_sql(items_query) %>%
    collect() %>%
    filter(rn == 1) %>%
    mutate(
      person_id = as.character(person_id),
      raw_score = unname(score_map[as.character(value_as_concept_id)]),
      score = ifelse(
        as.character(observation_source_concept_id) %in% reverse_items,
        reverse_max - raw_score, raw_score
      )
    )

  items_df %>%
    filter(!is.na(score)) %>%
    group_by(person_id) %>%
    summarise(n_items = n(), phenotype = sum(score), .groups = "drop") %>%
    filter(n_items == length(item_concept_ids)) %>%   # require the full scale
    select(person_id, phenotype)
}

# builds+writes one SDOH composite's modeling table -- same plausible-range +
# covariate-join + transform steps as alcohol_audit_c_score above
write_sdoh_composite_table <- function(phenotype_name, composite_df, demographics, keep_ids, covars) {
  row <- pheno_list_all %>% filter(phenotype_name == !!phenotype_name)
  stopifnot(nrow(row) == 1)

  df <- composite_df %>%
    inner_join(demographics, by = "person_id") %>%
    filter(person_id %in% keep_ids)

  range_result <- filter_plausible_range(df, "phenotype", as.numeric(row$plausible_min), as.numeric(row$plausible_max))
  df <- range_result$data %>%
    left_join(covars$pcs, by = "person_id") %>%
    left_join(covars$zip3, by = "person_id") %>%
    left_join(covars$ses, by = "person_id") %>%
    add_transformed_variant("phenotype")

  out_path <- file.path(MODELING_TABLE_DIR, paste0(phenotype_name, ".tsv"))
  write_tsv(df, out_path)
  message(sprintf("%s: %d excluded as implausible, %d written to %s", phenotype_name, range_result$n_excluded, nrow(df), out_path))
}

sdoh_demographics <- aou_sql(sprintf("
  SELECT
    person_id,
    CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
    CASE
      WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
      WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
      ELSE 'Other'
    END AS sex_at_birth
  FROM {CDR}.person
", REFERENCE_DATE)) %>% collect() %>% mutate(person_id = as.character(person_id))

sdoh_covars <- pull_covariates(keep_ids)

# PSS (Cohen's Perceived Stress Scale-10) -- confirmed via real {CDR}.concept
# labels in sdoh_environment_concept_search.ipynb
PSS_ITEM_IDS <- c(40192452, 40192381, 40192491, 40192419, 40192525, 40192506, 40192449, 40192445, 40192396, 40192462)  # cpss_1..10
PSS_SCORE_MAP <- c(`45876662` = 0, `45881665` = 1, `45882528` = 2, `45879226` = 3, `45884601` = 4, `903096` = NA_real_)
PSS_REVERSE_ITEMS <- c("40192419", "40192525", "40192449", "40192445")  # cpss_4, cpss_5, cpss_7, cpss_8

pss_df <- compute_sdoh_composite(PSS_ITEM_IDS, PSS_SCORE_MAP, PSS_REVERSE_ITEMS, reverse_max = 4)
write_sdoh_composite_table("pss_score", pss_df, sdoh_demographics, keep_ids, sdoh_covars)

# RAND MOS Social Support (8 items)
MOS_SS_ITEM_IDS <- c(40192442, 40192480, 40192388, 40192511, 40192439, 40192528, 40192399, 40192446)  # mos_ss_1..8
MOS_SS_SCORE_MAP <- c(`45884592` = 0, `45876996` = 1, `45879198` = 2, `45879199` = 3, `45883772` = 4, `903096` = NA_real_)

social_support_df <- compute_sdoh_composite(MOS_SS_ITEM_IDS, MOS_SS_SCORE_MAP)
write_sdoh_composite_table("social_support_score", social_support_df, sdoh_demographics, keep_ids, sdoh_covars)

# UCLA Loneliness Scale (8 items, non-sequential item numbers in this CDR) --
# confirmed 4-point (Never/Rarely/Sometimes/Often), not the published 3-point scale
UCLA_LS8_ITEM_IDS <- c(40192398, 40192501, 40192516, 40192390, 40192494, 40192507, 40192397, 40192504)  # ucla_ls8_{11,14,15,17,18,2,3,9}
UCLA_LS8_SCORE_MAP <- c(`45876662` = 0, `45876672` = 1, `45882528` = 2, `45884455` = 3, `903096` = NA_real_)

loneliness_df <- compute_sdoh_composite(UCLA_LS8_ITEM_IDS, UCLA_LS8_SCORE_MAP)
write_sdoh_composite_table("loneliness_score", loneliness_df, sdoh_demographics, keep_ids, sdoh_covars)

In [ ]:
# Everyday Discrimination Scale (eds_1..eds_9) -- excludes eds_follow_up_1(_xx),
# a different open-response "main reason" question, not part of the 9-item
# frequency scale
EDS_ITEM_IDS <- c(40192466, 40192489, 40192416, 40192490, 40192380, 40192395, 40192496, 40192519, 40192451)

# Not yet confirmed against real query results the way PSS/MOS-SS/UCLA-LS8 were
# (see sdoh_environment_concept_search.ipynb) -- this pulls the REAL distinct
# answer-option labels for these 9 items directly, so the map below is built
# from what's actually in this CDR, not assumed from the published EDS
# instrument.
eds_answer_options <- aou_sql(sprintf("
  SELECT DISTINCT o.value_as_concept_id, c.concept_name
  FROM {CDR}.observation o
  JOIN {CDR}.concept c ON o.value_as_concept_id = c.concept_id
  WHERE o.observation_source_concept_id IN (%s)
", paste(EDS_ITEM_IDS, collapse = ","))) %>% collect()
eds_answer_options

In [ ]:
# Builds EDS_SCORE_MAP by text-matching eds_answer_options's real concept_names
# against the published 6-point EDS response scale, rather than hardcoding
# concept_ids -- any label that doesn't match surfaces as a loud warning
# (inspect eds_answer_options above and extend EDS_LABEL_SCORES) rather than
# silently scoring as NA.
EDS_LABEL_SCORES <- c(
  "never" = 0,
  "less than once a year" = 1,
  "a few times a year" = 2,
  "a few times a month" = 3,
  "at least once a week" = 4,
  "almost every day" = 5,
  "pmi: skip" = NA_real_,
  "pmi: dont know" = NA_real_,
  "pmi: prefer not to answer" = NA_real_
)

eds_score_map_df <- eds_answer_options %>%
  mutate(label = tolower(trimws(concept_name)), score = unname(EDS_LABEL_SCORES[label]))

unmatched <- eds_score_map_df %>% filter(is.na(score) & !label %in% names(EDS_LABEL_SCORES))
if (nrow(unmatched) > 0) {
  warning(
    "EDS answer labels not recognized -- extend EDS_LABEL_SCORES above before trusting eds_score: ",
    paste(unmatched$concept_name, collapse = "; ")
  )
} else {
  EDS_SCORE_MAP <- setNames(eds_score_map_df$score, as.character(eds_score_map_df$value_as_concept_id))

  eds_df <- compute_sdoh_composite(EDS_ITEM_IDS, EDS_SCORE_MAP)
  write_sdoh_composite_table("eds_score", eds_df, sdoh_demographics, keep_ids, sdoh_covars)
}

## Run residualization

`run_residualization_from_tables()` reads `MODELING_TABLE_DIR`'s TSVs back
in and runs the full cross product of {phenotype} x {raw, invnorm} x
{covariate-set}, all exported -- no curation, every combination in
`pheno_list` x `covariate_sets` gets written out as an `FID IID Y` file
matching `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected format. No
BigQuery access needed -- this is the cell to re-run (on its own, without
re-running "Prepare modeling tables" above) when retuning
`covariate_sets`, `outlier_sd`, or which phenotypes to residualize.

A phenotype with no table in `MODELING_TABLE_DIR` (skipped above, or never prepared) is likewise skipped here rather than erroring on a missing file -- `message()`-printed, recorded in `combo_summary_table` with `status == "skipped: no modeling table found"`.

In [ ]:
result <- run_residualization_from_tables(pheno_list, MODELING_TABLE_DIR, covariate_sets, OUT_DIR)

## Diagnostics summary

In [ ]:
result$skew_summary_table   # per phenotype: skewness before/after the invnorm transform

In [ ]:
result$combo_summary_table  # per phenotype x variant x covariate-set: N retained, R^2